### This program calculates the optimal cost incurred to generate the most power

### The code mainly uses an optimisational paradigm, but also uses search and decision paradigms. 

In [1]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import sympy as sp
import sys

In [2]:
#Entering company name for labeling
print("CONSERVING POWER FLOW AT OPTIMAL COST")
comp_name = input("Please enter your company name")
while True:
    print()
    print("1. Find the optimal cost through a given function")
    print("2. Find the optimal cost through inputted data points")
    print("3. Quit") 
    c = int(input("What would you like to choose?"))
    if(c == 1):
        # Input the power function
        input_func = input("Enter the power function in terms of cost incurred as x: ")
        x = sp.Symbol("x")
        
        # Trying to convert the input given by the user to a sympy function
        try:
            opt_func = sp.sympify(input_func)
        except:
            print("The input is not convertible to a sympy expression.")
            continue
        
        # Generating the derivative and double derivative of the function by differentiating it.
        opt_diff = sp.diff(opt_func, x)
        opt_double_diff = sp.diff(opt_func, x, 2)
        
        # Finding the critical points by finding the roots of the first derivative of the function 
        crit_pts = sp.solve(opt_diff, x)
        
        # Converting the sympy functions into numpy functions for easier plotting
        opt_func_num = sp.lambdify(x, opt_func, "numpy")
        opt_diff_num = sp.lambdify(x, opt_diff, "numpy")
        opt_double_diff_num = sp.lambdify(x, opt_double_diff, "numpy")
        
        # Defining the range of plotting
        x_max=int(input("Enter the maximum value of cost"))
        x_vals = np.linspace(0, x_max, 500)
        
        # Creating the figure and subplots using add_subplot
        fig = plt.figure(figsize=(10, 12))
        
        # Plotting the original function
        ax1 = fig.add_subplot(311)
        ax1.plot(x_vals, opt_func_num(x_vals), label="Original Function")
        ax1.set_xlabel("Cost")
        ax1.set_ylabel("Power Generated")
        ax1.set_title(f"{comp_name}'s cost versus power function")
        ax1.legend()
        print()
        c = input("Would you like to see the first and second derivatives of the function? ")
        if(c.lower() == "yes"):
            # Plotting the first derivative
            ax2 = fig.add_subplot(312)
            if opt_diff.is_constant():
                # If the first derivative is a constant
                constant_value = float(opt_diff)
                ax2.plot(x_vals, np.ones_like(x_vals) * constant_value, label="First Derivative", color="orange")
            else:
                # If the first derivative is not a constant
                y_vals = opt_diff_num(x_vals)
                ax2.plot(x_vals, y_vals, label="First Derivative", color="orange")
            
            ax2.set_xlabel("Cost")
            ax2.set_ylabel("Slope")
            ax2.set_title("First Derivative")
            ax2.legend()
            
            # Plotting the second derivative
            ax3 = fig.add_subplot(313)
            if opt_double_diff.is_constant():
                # If the second derivative is a constant
                constant_value = float(opt_double_diff)
                ax3.plot(x_vals, np.ones_like(x_vals) * constant_value, label="Second Derivative", color="green")
            else:
                # If it's not a constant, evaluate the numerical function
                y_vals = opt_double_diff_num(x_vals)
                ax3.plot(x_vals, y_vals, label="Second Derivative", color="green")
            
            ax3.set_xlabel("Cost")
            ax3.set_ylabel("Curvature")
            ax3.set_title("Second Derivative")
            ax3.legend()
        
        print()
        print("Displaying the graphs: ")
        #Showing the plots
        plt.tight_layout()
        plt.show()
        
        # Finding the maxima out of the critical points
        optimal_point = None
        if (crit_pts != []):
            optimal_point = crit_pts[0]
            for i in crit_pts:
                if (opt_func.subs(x, i) < opt_func.subs(x, optimal_point)):
                    optimal_point = i
        
            # Checking the second derivative in order to confirm whether the critical point found is a maxima or a minima
            if(opt_double_diff.subs(x, optimal_point) < 0):
                print("The optimal cost generating the most power is:", optimal_point)
            else:
                print("Cannot compute optimal cost for this maximum power function")
        else:
            print("No critical points found.")
    elif(c == 2):
        print()
        # Collect x and y data points from the user
        print("Enter data points as x,y pairs where x is cost and y is power generated(while they are in same units respectively) Type 'done' when finished.")
        x = []
        y = []
        degree = -1
        while True:
            user_input = input("Enter x,y: ").strip()
            degree += 1
            if user_input.lower() == "done":
                break
            try:
                xi, yi = map(float, user_input.split(","))
                x.append(xi)
                y.append(yi)
            except ValueError:
                print("Invalid input. Please enter in the format x,y")
        
        # Ensure we have enough data points
        if len(x) < 2:
            print("You need at least two data points for polynomial fitting.")
        else:
            x = np.array(x)
            y = np.array(y)
        
            # Create the Vandermonde matrix
            X = np.vander(x, degree + 1, increasing=True)
        
            # Solve for coefficients using the normal equation: (X^T X) c = X^T y
            coefficients = np.linalg.inv(X.T @ X) @ X.T @ y
        
            # Define the polynomial function
            def polynomial(t):
                result = 0
                for i in range(len(coefficients)):
                    result += coefficients[i] * (t ** i)
                return result
        
            # Generate the polynomial string
            poly_str = " + ".join(
                [f"{coefficients[i]:.3f}*x^{i}" if i > 0 else f"{coefficients[i]:.3f}" for i in range(len(coefficients))]
            )
            print(f"\nFitted Polynomial Function:\n{poly_str}")
        
            # Define the derivative function manually
            def derivative(t):
                result = 0
                for i in range(1, len(coefficients)):  # Skip the 0th term (constant term)
                    result += i * coefficients[i] * (t ** (i - 1))
                return result
        
            # Define the second derivative manually
            def second_derivative(t):
                result = 0
                for i in range(2, len(coefficients)):  # Skip the first two terms
                    result += i * (i - 1) * coefficients[i] * (t ** (i - 2))
                return result
        
            # Find critical points (roots of the derivative)
            derivative_coeffs = [i * coefficients[i] for i in range(1, len(coefficients))]
            critical_points = np.roots(derivative_coeffs[::-1])
        
            # Filter critical points within the range of x values
            critical_points = [pt for pt in critical_points if min(x) <= pt <= max(x) and np.isreal(pt)]
            maxima = []
        
            # Check the second derivative to classify maxima
            for pt in critical_points:
                pt = np.real(pt)  # Ensure it's real
                if second_derivative(pt) < 0:  # Negative second derivative => Maximum
                    maxima.append((pt, polynomial(pt)))
        
            # Display maxima
            if maxima:
                print("The optimal cost generating the most power is:", value)
                print("\nMaxima:")
                for pt, value in maxima:
                    print(f"x = {pt:.3f}, y = {value:.3f}")
            else:
                print("Cannot compute optimal cost for this maximum power function")
        
            # Generate fitted values for plotting
            x_fit = np.linspace(min(x), max(x), 100)
            y_fit = [polynomial(t) for t in x_fit]
        
            # Plot the results
            plt.scatter(x, y, color='red', label='Data Points')
            plt.plot(x_fit, y_fit, label=f'Fitted Polynomial (degree {degree})', color='blue')
            if maxima:
                for mx, my in maxima:
                    plt.scatter(mx, my, color='green', label=f"{comp_name}'s maximum power output  at x={mx:.2f}")
            plt.legend()
            plt.title(f"{comp_name}'s cost versus power function")
            plt.xlabel("Cost")
            plt.ylabel("Power generated")
            plt.grid()
            plt.show()
    elif(c == 3):
        print()
        print("Thank you!")
        break


CONSERVING POWER FLOW AT OPTIMAL COST

1. Find the optimal cost through a given function
2. Find the optimal cost through inputted data points
3. Quit

Thank you!
